# IOAI — 2025 Selection 2 Letter Footprint (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os
os.makedirs('data', exist_ok=True)
print('Letter Footprint 은 외부 데이터가 없습니다(seed=42 망 재현). data/ 준비 완료.')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# The footprint of a letter — MAIO 2025 (Selection 2)

seed 42 로 고정된 망(`26 → 64 → 50`)이 있다. 입력은 알파벳 i번째 글자를 뜻하는 **one-hot 벡터 × 어떤 스칼라**
이다. 아래 4개의 `footprint`(망 출력)만 보고, 이를 만든 **네 글자**(소문자, 한 문자열로 이어서)를 찾아라.

`bias = 0` 이고 ReLU 는 양수 스칼라에 선형이므로, 각 글자의 출력은 스칼라에 **정비례**한다 → 글자마다
one-hot(v=1) 출력 방향과 footprint 를 비교(최소제곱)하면 글자와 스칼라를 정확히 복원할 수 있다.
결과는 `data/submission.csv`(`answer` 한 칸)에 저장한다.


In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import string

torch.manual_seed(42)                 # 시드 고정이 문제의 핵심 — 망이 결정적으로 재현된다.

class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)
        self._initialize_weights()
    def _initialize_weights(self):
        nn.init.normal_(self.fc1.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.fc2.weight, mean=0.0, std=0.02)
        nn.init.constant_(self.fc1.bias, 0.0)
        nn.init.constant_(self.fc2.bias, 0.0)
    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

model = SimpleNet(input_size=26, hidden_size=64, output_size=50).eval()

In [ ]:
footprints = torch.tensor([
[-0.0293,-0.1092,-0.0290,0.0885,0.0623,-0.1106,-0.0229,0.1987,-0.1062,0.1949,-0.0279,0.0705,-0.2228,-0.0478,-0.1745,0.1071,-0.0549,0.1062,0.1074,-0.0229,-0.1208,-0.0866,-0.1071,-0.0919,0.1597,0.1101,0.0335,0.0424,-0.1154,0.0375,0.0545,-0.2223,-0.0270,-0.0340,0.0261,0.1256,0.0372,-0.0898,-0.1757,0.1671,-0.1561,0.0908,0.0166,-0.0705,-0.0078,-0.0120,-0.0082,-0.0245,-0.1756,-0.0689],
[-0.1702,-0.2287,-0.0547,0.0448,0.2204,0.0330,-0.1223,0.0966,-0.0118,-0.0911,0.1477,0.1065,-0.3109,-0.0817,-0.0824,0.1230,0.1757,0.1231,-0.0162,-0.1775,-0.1278,-0.1683,0.1159,0.1903,-0.0053,0.0484,-0.0476,0.0044,-0.0514,-0.1619,-0.0399,0.0002,0.0555,-0.2482,-0.0462,-0.4026,-0.0799,-0.3325,-0.0719,0.2053,0.0951,0.2918,-0.0194,-0.0637,-0.1995,0.1005,-0.0509,-0.0357,-0.2760,-0.1030],
[-0.1158,-0.2736,-0.1917,0.1027,-0.0901,0.3210,-0.1885,0.1472,0.1963,-0.1349,-0.2345,0.5342,-0.2488,0.2003,-0.0306,-0.1891,0.1288,0.2327,0.0963,-0.1089,-0.2410,0.1069,-0.1464,-0.2085,0.0420,0.1267,-0.1581,-0.0614,0.0604,0.0312,-0.2559,0.0039,0.3445,-0.0059,0.0890,-0.0916,0.1148,0.0775,0.0633,0.1515,-0.1137,0.0521,-0.0418,0.0388,-0.0424,0.0375,-0.0586,0.0932,-0.0640,-0.1933],
[0.2047,-0.2309,-0.2069,-0.3843,0.0616,-0.0781,0.1256,0.2333,-0.0234,-0.0146,-0.3804,0.3934,-0.1173,-0.2266,-0.2054,0.0861,-0.0513,0.1966,0.1906,0.0125,-0.3607,-0.2924,-0.0591,-0.3109,0.3125,0.4452,-0.1773,-0.1590,-0.2283,-0.0456,-0.0041,-0.0896,0.1555,-0.1307,0.2646,-0.1352,0.1714,0.0815,0.3392,-0.2495,0.2008,-0.0400,0.0700,-0.1225,-0.3702,-0.2685,0.0006,-0.2181,-0.0386,-0.4234]])
print(footprints.shape)   # (4, 50)

In [ ]:
# ── 여기서부터 직접 풀어 보세요 ──────────────────────────────────────────
# 힌트: fc1.bias=0, fc2.bias=0 이고 ReLU 는 양수 스칼라에 homogeneous 하므로
#       model(v · e_i) = v · model(e_i)  →  각 글자의 단위출력 방향으로 footprint 를 설명할 수 있다.
#       글자 i 의 단위출력 base_i = model(torch.eye(26)[i]) 와 footprint 를 최소제곱으로 맞춰 보세요.

answer = "aaaa"       # TODO: 네 글자(소문자)를 찾아 이 문자열을 채우세요.

print("answer:", answer)

In [ ]:
import pandas as pd
pd.DataFrame({"answer": [answer]}).to_csv("submission.csv", index=False)
print("saved data/submission.csv")

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)